# Aprendendo tcpdump na prática

Este notebook roda dentro de um container Linux, então o `tcpdump` funciona
de forma nativa e idêntica à de qualquer servidor Linux real.

Vamos: (1) ver as interfaces disponíveis, (2) capturar tráfego gerado por nós
mesmos, (3) interpretar a saída, e (4) usar `scapy` para uma visão mais
estruturada dos pacotes.

In [ ]:
# 1. Ver interfaces de rede disponíveis dentro do container
!ip addr

Repare que dentro do container normalmente existe apenas `lo` (loopback) e
`eth0` (a interface virtual criada pelo Docker). É nela que vamos capturar.

In [ ]:
# 2. Captura básica: 10 pacotes, sem resolver nomes (-nn), formato compacto
!timeout 10 tcpdump -i eth0 -c 10 -nn

Nesse momento provavelmente não apareceu nada (ou pouca coisa), porque
ninguém gerou tráfego ainda. Vamos corrigir isso: capturar **enquanto**
geramos tráfego de propósito.

In [ ]:
# 3. Captura + geração de tráfego ao mesmo tempo
%%bash
(tcpdump -i eth0 -c 15 -nn -w /tmp/captura.pcap &)
sleep 1
curl -s https://google.com > /dev/null
sleep 2
echo "--- Leitura da captura ---"
tcpdump -r /tmp/captura.pcap -nn

## Interpretando a saída

Cada linha tem o formato:

`horário IP origem.porta > IP destino.porta: Flags [...], seq X, ack Y, win Z, length N`

Repare no início da captura o **three-way handshake do TCP**:

1. `Flags [S]` → SYN (cliente inicia conexão)
2. `Flags [S.]` → SYN-ACK (servidor responde)
3. `Flags [.]` → ACK (cliente confirma)

Depois disso vem o handshake TLS (porque é HTTPS), e só então os dados
de fato trafegam.

In [ ]:
# 4. Comparando: tráfego DNS (antes de qualquer TCP)
%%bash
(tcpdump -i eth0 -c 6 -nn 'udp port 53' -w /tmp/dns.pcap &)
sleep 1
dig +short example.com > /dev/null
sleep 2
tcpdump -r /tmp/dns.pcap -nn

Aqui usamos um **filtro BPF** (`udp port 53`) — o tcpdump aceita filtros
poderosos: `tcp`, `udp`, `port 443`, `host 8.8.8.8`, `src`, `dst`, e
combinações com `and`/`or`. Isso é essencial na prática, porque capturar
"tudo" em uma rede real é inviável.

In [ ]:
# 5. Ver o conteúdo bruto dos pacotes (-X mostra hex + ASCII)
%%bash
(tcpdump -i eth0 -c 5 -nn -X -w /tmp/http.pcap &)
sleep 1
curl -s http://neverssl.com > /dev/null
sleep 2
tcpdump -r /tmp/http.pcap -nn -X

Com HTTP puro (sem TLS) dá pra literalmente ler o conteúdo da requisição
no hexdump — ótimo gancho para explicar por que HTTPS existe.

In [ ]:
# 6. Visão estruturada com scapy (alternativa mais "pedagógica" ao texto puro)
from scapy.all import rdpcap

pacotes = rdpcap("/tmp/captura.pcap")
for pkt in pacotes:
    print(pkt.summary())

In [ ]:
# 7. Inspecionando um pacote específico camada por camada
pacotes[0].show()

## Exercícios sugeridos

- Troque o filtro para `icmp` e rode `!ping -c 4 8.8.8.8` numa célula em
  paralelo para ver o formato dos pacotes ICMP (ping).
- Capture só as portas 80 e 443 ao mesmo tempo com `'tcp port 80 or tcp port 443'`.
- Use `-v`, `-vv` e `-vvv` no tcpdump e compare o nível de detalhe.